In [17]:
import anatomist.api as ana
from soma.qt_gui.qtThread import QtThreadCall
from soma.qt_gui.qt_backend import Qt

a = ana.Anatomist()

from soma import aims
from scipy import ndimage
import numpy as np
import glob
import os
import json

from PIL import Image, ImageFont, ImageDraw

In [18]:
subject = "197550"
side = "R"

sources = glob.glob(f'/neurospin/dico/data/deep_folding/current/datasets/hcp/crops/2mm/*')
file_src = f'/neurospin/dico/data/deep_folding/current/datasets/hcp/skeletons/2mm/{side}/{side}resampled_skeleton_{subject}.nii.gz'

In [19]:
def to_bucket(obj):
    if obj.type() == obj.BUCKET:
        return obj
    avol = a.toAimsObject(obj)
    c = aims.Converter(intype=avol, outtype=aims.BucketMap_VOID)
    abck = c(avol)
    bck = a.toAObject(abck)
    bck.releaseAppRef()
    return bck


def crop_mask(file_src, file_cropped, mask):
    """Crops according to mask"""
    volume = aims.read(file_src)
    print(np.count_nonzero(volume.np))
    if mask:
        mask = aims.read(mask)
        arr = volume.np
        arr_mask = np.asarray(mask)
        arr[arr_mask == 0] = 0
        print(np.count_nonzero(volume.np))
    aims.write(volume, file_cropped)

def build_gradient(pal):
    """Build a gradient palette for Anatomist visualization."""
    gw = ana.cpp.GradientWidget(None, 'gradientwidget', pal.header()['palette_gradients'])
    gw.setHasAlpha(True)
    nc = pal.shape[0]
    rgbp = gw.fillGradient(nc, True)
    rgb = rgbp.data()
    npal = pal.np['v']
    pb = np.frombuffer(rgb, dtype=np.uint8).reshape((nc, 4))
    npal[:, 0, 0, 0, :] = pb
    # Convert BGRA to RGBA
    npal[:, 0, 0, 0, :3] = npal[:, 0, 0, 0, :3][:, ::-1]
    pal.update()

def create_grid(image_files, n_cols, out_path, title=None, subject_names=None):
    # load all images
    imgs = [Image.open(f) for f in image_files]
    # calculate max width and height
    w = max(im.width for im in imgs)
    h = max(im.height for im in imgs)
    # calculate number of rows
    n_rows = (len(imgs) + n_cols - 1) // n_cols

    title_h = 0
    legend_h = 0
    font_size = 36
    if title or subject_names:
        font = ImageFont.truetype("DejaVuSans.ttf", font_size)
        if title:
            title_h = font.getbbox(title)[3] - font.getbbox(title)[1] + 10  # add margin
        if subject_names:
            legend_h = font.getbbox("Test")[3] - font.getbbox("Test")[1] + 15  # idem

    # create a new blank image with space for title and legend
    grid = Image.new('RGB', (n_cols * w, n_rows * (h + legend_h) + title_h), (255, 255, 255))
    draw = ImageDraw.Draw(grid)

    # Draw title
    if title:
        bbox = draw.textbbox((0, 0), title, font=font)  
        text_w = bbox[2] - bbox[0]
        x = (grid.width - text_w) // 2
        draw.text((x, 5), title, fill=(0, 0, 0), font=font)

    # Paste images and draw subject names
    for idx, im in enumerate(imgs):
        i, j = divmod(idx, n_cols)
        x0 = j * w
        y0 = title_h + i * (h + legend_h)
        grid.paste(im, (x0, y0))

        if subject_names:
            subj = subject_names[idx]
            text_bbox = draw.textbbox((0, 0), subj, font=font)
            text_w = text_bbox[2] - text_bbox[0]
            draw.text((x0 + (w - text_w) // 2, y0 + h-50), subj, fill=(0, 0, 0), font=font)

    grid.save(out_path)
    print(f"Snapshot of the block available at {out_path}")

In [20]:
PATH_LIST_REGIONS = "/neurospin/dico/data/deep_folding/current/sulci_regions_champollion_V1.json"
with open(PATH_LIST_REGIONS) as f:
    d = json.load(f)

list_regions = [w.replace('_left', '').replace('_right', '') for w in list(d['brain'].keys())]
list_regions = list(set(list_regions))
len(list_regions)

28

In [21]:
"""
bv bash
cd /volatile/ad279118/deep_folding
. venv/bin/activate
pip install -e .
cd /volatile/ad279118/deep_folding/deep_folding/brainvisa/utils
python3 convert_volume_to_bucket.py -s '/volatile/ad279118/Figures_report/197550' -t '/volatile/ad279118/Figures_report/197550'
"""

"\nbv bash\ncd /volatile/ad279118/deep_folding\n. venv/bin/activate\npip install -e .\ncd /volatile/ad279118/deep_folding/deep_folding/brainvisa/utils\npython3 convert_volume_to_bucket.py -s '/volatile/ad279118/Figures_report/197550' -t '/volatile/ad279118/Figures_report/197550'\n"

In [22]:
w = a.createWindow("3D")
w2 = a.createWindow("3D")
dic_windows = {}

### To load the white mesh of the subject

In [23]:
# to plot the whole sulcal skeleton of the subject
dic_windows[f'source_{subject}'] = a.loadObject(file_src)
dic_windows[f'source_{subject}'].loadReferentialFromHeader()
dic_windows[f'fusion_{subject}'] = a.fusionObjects(objects=[dic_windows[f'source_{subject}']], method='VolumeRenderingFusionMethod')
w.addObjects(dic_windows[f'fusion_{subject}'])

# to plot the white mesh of the same subject
path_to_t1mri = f'/neurospin/dico/data/bv_databases/human/not_labeled/hcp/hcp/{subject}/t1mri/BL'
dic_windows[f'white_{subject}'] = a.loadObject(f'{path_to_t1mri}/default_analysis/segmentation/mesh/{subject}_{side}white.gii')
dic_windows[f'white_{subject}'].loadReferentialFromHeader()

nifti transfo: 1


ATransformSet::unregisterObserver: ref 0x59c63bb7e920 not found


nifti transfo: 2


### To load the buckets of each masked region

In [24]:
for source in sources:
    region = source.replace('/neurospin/dico/data/deep_folding/current/datasets/hcp/crops/2mm/', '')
    if region in ['CENTRAL']:
        continue
    mask_path = f'{source}/mask/{side}mask_skeleton.nii.gz'
    #file_cropped = f'/volatile/ad279118/Figures_report/{subject}/{subject}_{region}_{side}_cropped_skeleton.nii.gz'
    #crop_mask(file_src, file_cropped, mask_path)
    #dic_windows[f'vol_{region}'] = aims.read(file_cropped)
    #dic_windows[f'a_obj_{region}'] = a.toAObject(dic_windows[f'vol_{region}'])
    #dic_windows[f'fusion_{region}'] = a.fusionObjects(objects=[dic_windows[f'a_obj_{region}']], method='VolumeRenderingFusionMethod')
    #w.addObjects(dic_windows[f'fusion_{region}'])

    #
    path_to_bck = f"/volatile/ad279118/Figures_report/197550/{subject}_{region}_{side}_cropped_skeleton.bck"
    dic_windows[f'bcks_{region}'] = a.loadObject(path_to_bck)
    dic_windows[f'bcks_{region}'].loadReferentialFromHeader()
    w2.addObjects(dic_windows[f'bcks_{region}'])

Position : 57.0469, 106.189, 101.986, 0
Position : 60.9875, 114.841, 108.378, 0
Position : 38.8995, 96.0703, 76.682, 0
Position : 44.8994, 99.7548, 74.0547, 0
no position could be read at 258, 193
Position : 42.9506, 67.4991, 108.493, 0
Position : 32.0101, 101.194, 133.928, 0
Position : 42.8606, 120.95, 145.547, 0
Position : 32.8606, 98.0763, 110.828, 0
Position : 34.8606, 79.997, 101.746, 0
Position : 38.8606, 99.5324, 118.002, 0
no position could be read at 284, 285
no position could be read at 284, 285
Position : 52.8606, 81.6281, 118.652, 0
Position : 50.8606, 90.735, 134.099, 0
Position : 52.8606, 81.7428, 135.079, 0
Position : 36.8606, 78.2097, 101.5, 0


QLayout: Attempting to add QLayout "" to QWidget "", which already has a layout


Position : 52.8748, 111.127, 57.9583, 0
Position : 70.9087, 77.5249, 56.0959, 0
Position : 54.8881, 85.1272, 117.06, 0


qt.qpa.xcb: QXcbConnection: XCB error: 3 (BadWindow), sequence: 11826, resource id: 4202467, major code: 40 (TranslateCoords), minor code: 0
qt.qpa.xcb: QXcbConnection: XCB error: 3 (BadWindow), sequence: 12045, resource id: 4202498, major code: 40 (TranslateCoords), minor code: 0
qt.qpa.xcb: QXcbConnection: XCB error: 3 (BadWindow), sequence: 12208, resource id: 4202607, major code: 40 (TranslateCoords), minor code: 0


Position : 57.9104, 83.1272, 109.991, 0


### To load the reconstructed hemisphere for a list of subjects
Needs the code decode_global_brain.py to have run before

In [32]:
nb_columns = 4
block = a.createWindowsBlock(nb_columns)
dic_windows2 = {}
pal = a.createPalette('VR-palette')
pal.header()['palette_gradients'] = "1;1#0;1;1;0#0.994872;0#0;0;0.635897;0.266667;1;1"
# 0;1;0.182051;1;0.248718;0;0.628205;0;0.75641;1;1;1#0;1;0.171795;1;0.235897;0;0.487179;0;0.646154;0.977778;1;1#0.220513;1;0.24359;0.822222;0.487179;0.822222;1;1#0;0;0.148718;1;0.828205;1;1;0
build_gradient(pal)
subjects = [
    100206,
    100307,
    100408,
    100610,
    101006,
    101107,
    101309,
    101410,
]

for subject in subjects:
    file_src = f'/neurospin/dico/data/deep_folding/current/datasets/hcp/skeletons/2mm/{side}/{side}resampled_skeleton_{subject}.nii.gz'
    file_recon = f"/volatile/ad279118/Figures_report/global_reconstruction/{side}_{subject}_decoded.nii.gz"
    # load the input 
    dic_windows2[f'w_init_{subject}'] = a.createWindow('3D', block=block)
    dic_windows2[f'obj_init_{subject}'] = a.loadObject(file_src)
    #dic_windows2[f'buck_init_{subject}'] = to_bucket(dic_windows2[f'obj_init_{subject}'])
    dic_windows2[f'fusion__init_{subject}'] = a.fusionObjects(objects=[dic_windows2[f'obj_init_{subject}']], 
                                                            method='VolumeRenderingFusionMethod')
    dic_windows2[f'w_init_{subject}'].addObjects(dic_windows2[f'fusion__init_{subject}'])

    # load the reconstruction
    dic_windows2[f'w_recon_{subject}'] = a.createWindow('3D', block=block)
    dic_windows2[f'obj_recon_{subject}'] = a.loadObject(file_recon)
    dic_windows2[f'fusion_recon_{subject}'] = a.fusionObjects(objects=[dic_windows2[f'obj_recon_{subject}']], 
                                                                        method='VolumeRenderingFusionMethod')
    dic_windows2[f'fusion_recon_{subject}'].setPalette('VR-palette', 
                                                       minVal=0, 
                                                       maxVal=0.5, 
                                                       absoluteMode=True)
    dic_windows2[f'w_recon_{subject}'].addObjects(dic_windows2[f'fusion_recon_{subject}'])
    #dic_windows2[f'w_init_{subject}'].addObjects(dic_windows2[f'fusion_recon_{subject}'])



no position could be read at 218, 144
no position could be read at 196, 150


### To save all the individual images

In [33]:
snapshot = True
image_files= []
if snapshot:
    for subject in subjects:
        save_dir = "/volatile/ad279118/Figures_report/global_reconstruction"
        dic_windows2[f'w_recon_{subject}'].setHasCursor(0)
        recon_fname = f"{subject}_{side}_global_reconstruction.png"
        recon_img_path = os.path.join(save_dir, recon_fname)
        dic_windows2[f'w_recon_{subject}'].snapshot(recon_img_path, width=1200, height=900)

        init_fname = f"{subject}_{side}_global_input.png"
        init_img_path = os.path.join(save_dir, init_fname)
        dic_windows2[f'w_init_{subject}'].snapshot(init_img_path, width=1200, height=900)
        image_files.append(init_img_path)
        image_files.append(recon_img_path)

### To save all the individual images within a grid

In [34]:
create_grid(image_files,4,f"{save_dir}/grid_2.png", title='Noillopmahc')

Snapshot of the block available at /volatile/ad279118/Figures_report/global_reconstruction/grid_2.png
